# Pairwise label consistency

This notebook loads the two extracted ranking exports, matches the same video pairs by filename, and computes one metric:

- `label_match_pct`: percent of matching pairs where both exports prefer the same video.


In [9]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
EXPORT_DIRS = [
    REPO_ROOT / "notebooks" / "data" / "label-consistency-test" / "ranks_1",
    REPO_ROOT / "notebooks" / "data" / "label-consistency-test" / "ranks_2",
]

for path in EXPORT_DIRS:
    if not path.exists():
        raise FileNotFoundError(path)

display(pd.DataFrame({"export_dir": [str(path) for path in EXPORT_DIRS]}))


,export_dir
0,/workspace/notebooks/data/label-consistency-te...
1,/workspace/notebooks/data/label-consistency-te...


In [10]:
def clip_name(value: str) -> str:
    text = str(value)
    if "://" in text:
        text = text.split("://", 1)[1]
    return Path(text).name


def load_export(export_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted((export_dir / "ranks").glob("*")):
        with path.open() as f:
            row = json.load(f)

        if not row.get("result"):
            continue

        left = row["task"]["data"]["left"]
        right = row["task"]["data"]["right"]
        pref = row["result"][0]["value"]["selected"]

        left_name = clip_name(left)
        right_name = clip_name(right)
        pair_key = tuple(sorted([left_name, right_name]))
        preferred_clip = left_name if pref == "left" else right_name

        rows.append(
            {
                "annotation_id": row["id"],
                "pair_key": pair_key,
                "left_name": left_name,
                "right_name": right_name,
                "preferred_clip": preferred_clip,
                "trick": row["task"]["data"].get("trick"),
            }
        )

    return pd.DataFrame(rows)


export_1 = load_export(EXPORT_DIRS[0])
export_2 = load_export(EXPORT_DIRS[1])

print(f"export_1 rows: {len(export_1)}")
print(f"export_2 rows: {len(export_2)}")
display(export_1.head())


export_1 rows: 282
export_2 rows: 50


,annotation_id,pair_key,left_name,right_name,preferred_clip,trick
0,1132,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-02 19-14-55 5695-00.05.54.650-00.05.58.8...,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,ao-soul
1,1133,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-02 19-14-55 5695-00.05.54.650-00.05.58.8...,25-11-07 15-47-00 5699-00.10.23.670-00.10.29.5...,25-11-07 15-47-00 5699-00.10.23.670-00.10.29.5...,ao-soul
2,1134,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-02 19-14-55 5695-00.05.54.650-00.05.58.8...,25-11-07 16-17-23 5702-00.02.36.390-00.02.42.8...,25-11-07 16-17-23 5702-00.02.36.390-00.02.42.8...,ao-soul
3,1135,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-02 19-14-55 5695-00.05.54.650-00.05.58.8...,25-11-17 21-54-30 5744-00.08.02.728-00.08.06.8...,25-11-17 21-54-30 5744-00.08.02.728-00.08.06.8...,ao-soul
4,1136,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-02 19-14-55 5695-00.05.54.650-00.05.58.8...,25-12-01 21-34-32 5970-00.05.49.354-00.05.54.4...,25-12-01 21-34-32 5970-00.05.49.354-00.05.54.4...,ao-soul


In [11]:
comparison = export_1.merge(
    export_2,
    on="pair_key",
    how="inner",
    suffixes=("_1", "_2"),
)

comparison["label_match"] = comparison["preferred_clip_1"] == comparison["preferred_clip_2"]

summary = pd.DataFrame(
    [
        {"metric": "matching_pairs", "value": len(comparison)},
        {
            "metric": "label_match_pct",
            "value": round(100 * comparison["label_match"].mean(), 2) if len(comparison) else 0.0,
        },
    ]
)

display(summary)
display(
    comparison[
        [
            "pair_key",
            "preferred_clip_1",
            "preferred_clip_2",
            "label_match",
        ]
    ].head(20)
)


,metric,value
0,matching_pairs,50.0
1,label_match_pct,90.0


,pair_key,preferred_clip_1,preferred_clip_2,label_match
0,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,True
1,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-07 15-47-00 5699-00.10.23.670-00.10.29.5...,25-11-07 15-47-00 5699-00.10.23.670-00.10.29.5...,True
2,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-07 16-17-23 5702-00.02.36.390-00.02.42.8...,25-11-07 16-17-23 5702-00.02.36.390-00.02.42.8...,True
3,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-11-17 21-54-30 5744-00.08.02.728-00.08.06.8...,25-11-02 19-14-55 5695-00.05.54.650-00.05.58.8...,False
4,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-12-01 21-34-32 5970-00.05.49.354-00.05.54.4...,25-12-01 21-34-32 5970-00.05.49.354-00.05.54.4...,True
5,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-12-01 21-34-32 5970-00.07.46.663-00.07.52.8...,25-12-01 21-34-32 5970-00.07.46.663-00.07.52.8...,True
6,(25-11-02 19-14-55 5695-00.05.54.650-00.05.58....,25-12-02 20-41-10 5974-00.10.11.153-00.10.14.7...,25-12-02 20-41-10 5974-00.10.11.153-00.10.14.7...,True
7,(25-11-02 19-14-55 5695-00.13.15.752-00.13.19....,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,True
8,(25-11-02 19-14-55 5695-00.13.15.752-00.13.19....,25-12-01 21-34-32 5970-00.05.49.354-00.05.54.4...,25-12-01 21-34-32 5970-00.05.49.354-00.05.54.4...,True
9,(25-11-02 19-14-55 5695-00.13.15.752-00.13.19....,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,25-11-02 19-14-55 5695-00.13.15.752-00.13.19.2...,True
